# Notebook 07 — Nemotron RAG Pipeline & Evaluation

**RHOAI 3.5 | Nemotron LLM | MLflow Experiment Tracking**

Build and evaluate a Retrieval-Augmented Generation (RAG) pipeline where Nemotron
generates answers grounded in a knowledge base, with every configuration tracked in MLflow.

1. Build a TF-IDF knowledge base (no external embedding API — works on disconnected clusters)
2. RAG pipeline with full MLflow tracing (retrieve → generate)
3. Systematic evaluation: compare `top_k` values and system prompt variants
4. Compare configurations in the MLflow Dashboard (nested runs + parallel coordinates)
5. Log the best configuration as a reusable artifact

---
**Prerequisites:** Notebooks 01 and 06 completed.
**No new packages** — uses `scikit-learn` TF-IDF (pre-installed in RHOAI workbench images).

## Section 1 — Setup

In [ ]:
import mlflow
import mlflow.tracking
from mlflow.entities import SpanType
import requests, os, time, uuid, json, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
requests.packages.urllib3.disable_warnings(
    requests.packages.urllib3.exceptions.InsecureRequestWarning
)

LLM_BASE_URL = 'https://llm-base-url.com' #replace this url
MODEL_ID    = 'nemotron-3-5-lightning'   # update if needed
EXPERIMENT_NAME   = 'rhoai-nemotron-rag-eval'

mlflow.set_experiment(EXPERIMENT_NAME)

SA_TOKEN_PATH = '/var/run/secrets/kubernetes.io/serviceaccount/token'
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        INFER_TOKEN = f.read().strip()
else:
    INFER_TOKEN = os.environ.get('OC_TOKEN', '')

HEADERS = {'Content-Type': 'application/json',
           'Authorization': f'Bearer {INFER_TOKEN}'}

print(f'Experiment  : {EXPERIMENT_NAME}')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

In [ ]:
# ── Endpoint discovery ────────────────────────────────────────────────
# Probes the service to detect protocol (OpenAI-compatible vs KServe v2)
# and auto-confirms the model name so RAG works on both vLLM and TGI runtimes.
API_FORMAT    = 'openai'   # 'openai' | 'kserve_v2'
CHAT_ENDPOINT = f'{LLM_BASE_URL}/v1/chat/completions'  # default

def _probe_nemotron():
    global MODEL_ID, API_FORMAT, CHAT_ENDPOINT

    # 1. Try OpenAI-compatible /v1/models
    try:
        r = requests.get(f'{LLM_BASE_URL}/v1/models',
                         headers=HEADERS, verify=False, timeout=10)
        if r.status_code == 200:
            models = [m['id'] for m in r.json().get('data', [])]
            print(f'OpenAI /v1/models → {models}')
            if models and MODEL_ID not in models:
                MODEL_ID = models[0]
                print(f'  Auto-updated MODEL_ID → {MODEL_ID}')
            CHAT_ENDPOINT = f'{LLM_BASE_URL}/v1/chat/completions'
            API_FORMAT    = 'openai'
            return
        print(f'/v1/models returned {r.status_code} — trying KServe v2...')
    except requests.exceptions.RequestException as e:
        print(f'/v1/models probe error: {e}')

    # 2. KServe v2 /v2/health/ready
    try:
        r = requests.get(f'{LLM_BASE_URL}/v2/health/ready',
                         headers=HEADERS, verify=False, timeout=10)
        if r.status_code == 200:
            print('/v2/health/ready → 200')
            CHAT_ENDPOINT = f'{LLM_BASE_URL}/v2/models/{MODEL_ID}/infer'
            API_FORMAT    = 'kserve_v2'
            return
    except requests.exceptions.RequestException as e:
        print(f'/v2/health/ready probe error: {e}')

    print('Warning: endpoint probe inconclusive — defaulting to OpenAI format.')
    print('  If you see 404s run:  oc get route -n <namespace> | grep nemotron')

_probe_nemotron()
print(f'  MODEL_ID : {MODEL_ID}')
print(f'  API_FORMAT     : {API_FORMAT}')
print(f'  CHAT_ENDPOINT  : {CHAT_ENDPOINT}')

## Section 2 — Knowledge Base

A TF-IDF retriever requires no external API or vector database.
Replace `KNOWLEDGE_BASE` with your own document chunks in production.

In [ ]:
KNOWLEDGE_BASE = [
    {'id': 'rhoai-overview', 'title': 'Red Hat OpenShift AI Overview',
     'content': ('Red Hat OpenShift AI (RHOAI) is a platform for data scientists and ML engineers '
                 'to develop, train, and deploy machine learning models on OpenShift. '
                 'RHOAI 3.5 includes managed workbenches, model serving, pipelines, and MLflow tracking.')},
    {'id': 'mlflow-auth', 'title': 'MLflow Authentication in RHOAI',
     'content': ('Every MLflow API call in RHOAI is authorized by Kubernetes RBAC. '
                 'Workbenches receive automatic env injection: MLFLOW_TRACKING_URI and '
                 'MLFLOW_TRACKING_AUTH=kubernetes-namespaced. No MLflow user database exists.')},
    {'id': 'mlflow-experiments', 'title': 'MLflow Experiments and Runs',
     'content': ('An MLflow experiment is a named collection of runs. A run represents one '
                 'execution of a training or evaluation job, recording parameters, metrics, '
                 'artifacts, and tags. Runs are searchable via mlflow.search_runs().')},
    {'id': 'mlflow-registry', 'title': 'MLflow Model Registry and Aliases',
     'content': ('The MLflow Model Registry stores versioned trained models. In MLflow 3.x, '
                 'aliases replace deprecated stage names. Load by alias: '
                 'mlflow.sklearn.load_model("models:/model@champion"). '
                 'Aliases allow zero-downtime promotion without changing inference code.')},
    {'id': 'mlflow-tracing', 'title': 'MLflow GenAI Tracing',
     'content': ('MLflow 3.x includes a Tracing API for LLM applications. '
                 'Use @mlflow.trace(span_type=SpanType.LLM) to capture inputs, outputs, '
                 'and timing. RHOAI 3.5 Dashboard shows traces in GenAI mode with '
                 'Overview, Traces, and Sessions tabs.')},
    {'id': 'trace-archival', 'title': 'Trace Archival',
     'content': ('Trace Archival moves span payloads from PostgreSQL to S3 on a schedule. '
                 'Enable with spec.traceArchival.enabled=true in the MLflowTrackingServer CR. '
                 'Archived traces remain viewable; span-content search filters stop working '
                 'on archived data — only timestamp, status, and tag filters remain.')},
    {'id': 'model-serving', 'title': 'RHOAI Model Serving',
     'content': ('RHOAI Model Serving uses KServe to deploy models as InferenceService resources. '
                 'vLLM is the common runtime for LLMs, exposing an OpenAI-compatible API '
                 'at /v1/chat/completions. Auth uses OpenShift Bearer tokens.')},
    {'id': 'disconnected', 'title': 'Disconnected Cluster Support',
     'content': ('On air-gapped clusters, bake Python packages into a custom workbench image '
                 'using pip install --no-index --find-links from pre-downloaded wheels. '
                 'Push to the internal registry and register in RHOAI Dashboard > Settings.')},
    {'id': 'nested-runs', 'title': 'Nested Runs for Hyperparameter Search',
     'content': ('Use mlflow.start_run(nested=True) to create child runs under a parent. '
                 'The parent represents the full search; children are individual configs. '
                 'Query children with filter_string on tags.mlflow.parentRunId.')},
    {'id': 'autolog', 'title': 'MLflow Autologging',
     'content': ('mlflow.sklearn.autolog() captures fit() parameters, metrics, model, '
                 'and artifacts automatically. Enable before training; disable after '
                 'to avoid interfering with manual logging. Supported for sklearn, '
                 'PyTorch, TensorFlow, and XGBoost.')},
]

print(f'Knowledge base: {len(KNOWLEDGE_BASE)} documents')
for doc in KNOWLEDGE_BASE:
    print(f'  [{doc["id"]}] {doc["title"]}')

In [ ]:
# Build TF-IDF index over the knowledge base
corpus    = [doc['content'] for doc in KNOWLEDGE_BASE]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f'TF-IDF index: {tfidf_matrix.shape[0]} docs, {tfidf_matrix.shape[1]} features')


@mlflow.trace(span_type=SpanType.RETRIEVER, name='tfidf-retrieve')
def retrieve(query: str, top_k: int = 3) -> list:
    """Retrieve top_k documents by TF-IDF cosine similarity."""
    q_vec  = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = scores.argsort()[-top_k:][::-1]
    results = [{**KNOWLEDGE_BASE[i], 'score': float(scores[i])}
               for i in top_idx if scores[i] > 0]

    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute('retriever.top_k',   top_k)
        span.set_attribute('retriever.doc_ids', [r['id'] for r in results])
        span.set_attribute('retriever.scores',  [round(r['score'], 4) for r in results])
    return results


# Quick smoke test
test_docs = retrieve('how does RHOAI authentication work', top_k=2)
print(f'\nTest retrieval (top 2):')
for d in test_docs:
    print(f'  [{d["score"]:.3f}] {d["title"]}')

## Section 3 — RAG Pipeline with Nemotron

The pipeline: **retrieve** relevant documents → **build context** → **generate** with Nemotron.
Each function is decorated with `@mlflow.trace` so the full call tree appears in the dashboard.

In [ ]:
def _call_nemotron(messages, temperature, max_tokens):
    """Send a generation request; handles OpenAI-compatible and KServe v2 protocols."""
    if API_FORMAT == 'openai':
        resp = requests.post(
            CHAT_ENDPOINT, headers=HEADERS, verify=False, timeout=60,
            json={'model': MODEL_ID, 'messages': messages,
                  'temperature': temperature, 'max_tokens': max_tokens},
        )
        resp.raise_for_status()
        data   = resp.json()
        choice = data['choices'][0]
        return {
            'answer'       : choice['message'].get('content') or '',
            'finish_reason': choice.get('finish_reason', ''),
            'usage'        : data.get('usage', {}),
        }
    else:   # kserve_v2
        prompt = '\n'.join(
            f"{m['role'].upper()}: {m['content']}" for m in messages
        )
        resp = requests.post(
            CHAT_ENDPOINT, headers=HEADERS, verify=False, timeout=60,
            json={
                'inputs': [{'name': 'text_input', 'shape': [1],
                            'datatype': 'BYTES', 'data': [prompt]}],
                'parameters': {'temperature': temperature,
                               'max_new_tokens': max_tokens},
            },
        )
        resp.raise_for_status()
        outputs = resp.json().get('outputs', [{}])
        answer  = outputs[0].get('data', [''])[0] if outputs else ''
        return {'answer': answer, 'finish_reason': '', 'usage': {}}


@mlflow.trace(span_type=SpanType.LLM, name='nemotron-generate')
def generate(context: str, question: str, system_prompt: str,
             temperature: float = 0.6, max_tokens: int = 400) -> dict:
    """Generate an answer from Nemotron given retrieved context."""
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': f'Context:\n{context}\n\nQuestion: {question}'},
    ]

    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute('llm.model',         MODEL_ID)
        span.set_attribute('llm.temperature',   temperature)
        span.set_attribute('llm.context_chars', len(context))
        span.set_attribute('llm.api_format',    API_FORMAT)

    t0     = time.time()
    result = _call_nemotron(messages, temperature, max_tokens)
    latency_ms = int((time.time() - t0) * 1000)
    usage      = result['usage']

    if span:
        span.set_attribute('llm.prompt_tokens',     usage.get('prompt_tokens', 0))
        span.set_attribute('llm.completion_tokens', usage.get('completion_tokens', 0))
        span.set_attribute('llm.total_tokens',      usage.get('total_tokens', 0))
        span.set_attribute('llm.latency_ms',        latency_ms)
        span.set_attribute('llm.finish_reason',     result.get('finish_reason', ''))

    return {'answer': result['answer'], 'usage': usage, 'latency_ms': latency_ms}


@mlflow.trace(span_type=SpanType.CHAIN, name='rag')
def rag(question: str, system_prompt: str, top_k: int = 3,
        temperature: float = 0.6) -> dict:
    """Full RAG pipeline: retrieve → build context → generate."""
    docs    = retrieve(question, top_k=top_k)
    context = '\n\n'.join(f'[{d["title"]}]\n{d["content"]}' for d in docs)
    result  = generate(context, question, system_prompt, temperature=temperature)
    return {
        'answer'        : result['answer'],
        'retrieved_docs': [d['id'] for d in docs],
        'context_chars' : len(context),
        'usage'         : result['usage'],
        'latency_ms'    : result['latency_ms'],
    }

print('rag() pipeline ready.')

In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    'You are an expert on Red Hat OpenShift AI and MLflow. '
    'Answer questions using only the provided context. '
    'If the context does not contain enough information, say so. '
    'Be concise and precise.'
)

test_question = 'How do I deploy MLflow for production use on RHOAI?'

with mlflow.start_run(run_name='rag-single-test'):
    mlflow.set_tags({'notebook': '07_nemotron_rag_eval', 'model': MODEL_ID})
    mlflow.log_params({'top_k': 3, 'temperature': 0.6, 'question': test_question})

    result = rag(test_question, DEFAULT_SYSTEM_PROMPT, top_k=3, temperature=0.6)

    mlflow.log_metrics({
        'prompt_tokens'    : result['usage'].get('prompt_tokens', 0),
        'completion_tokens': result['usage'].get('completion_tokens', 0),
        'total_tokens'     : result['usage'].get('total_tokens', 0),
        'latency_ms'       : result['latency_ms'],
        'context_chars'    : result['context_chars'],
        'answer_length'    : len(result['answer']),
        'retrieved_docs'   : len(result['retrieved_docs']),
    })

print(f'Q: {test_question}')
print(f'Retrieved: {result["retrieved_docs"]}')
print(f'A: {result["answer"]}')
print(f'Latency: {result["latency_ms"]} ms   Tokens: {result["usage"]}')

## Section 4 — Systematic Evaluation

Grid search over **top_k** (1, 2, 3) × **system prompt variant** (concise, detailed, structured)
across 5 evaluation questions. Each configuration is a nested child run under one parent.

The RHOAI Dashboard parallel coordinates plot makes it easy to see which combination
produces the best latency / answer completeness trade-off.

In [ ]:
EVAL_QUESTIONS = [
    'What authentication method does RHOAI use for MLflow API calls?',
    'How do MLflow aliases differ from legacy stage names?',
    'What is trace archival and what happens to search filters after archival?',
    'How do you install MLflow in a disconnected cluster with no internet access?',
    'What is the difference between an MLflow experiment and a run?',
]

SYSTEM_PROMPT_VARIANTS = {
    'concise': (
        'You are a technical assistant. Answer using only the provided context. '
        'Be brief — 2 to 3 sentences maximum.'
    ),
    'detailed': (
        'You are an expert on Red Hat OpenShift AI and MLflow. '
        'Answer using only the provided context. '
        'Provide a thorough explanation with relevant technical details.'
    ),
    'structured': (
        'You are a technical assistant for RHOAI and MLflow. '
        'Answer using only the provided context. '
        'Structure your answer: (1) direct answer, (2) explanation, (3) example if applicable.'
    ),
}

TOP_K_VALUES = [1, 2, 3]

total_runs = len(EVAL_QUESTIONS) * len(SYSTEM_PROMPT_VARIANTS) * len(TOP_K_VALUES)
print(f'Evaluation grid: {len(EVAL_QUESTIONS)} questions × '
      f'{len(SYSTEM_PROMPT_VARIANTS)} prompts × {len(TOP_K_VALUES)} top_k = {total_runs} child runs')

In [ ]:
print('Running evaluation grid...')

with mlflow.start_run(run_name='rag-eval-sweep') as parent_run:
    mlflow.set_tags({'notebook': '07_nemotron_rag_eval', 'eval_type': 'grid', 'model': MODEL_ID})
    mlflow.log_params({'n_questions': len(EVAL_QUESTIONS), 'top_k_values': str(TOP_K_VALUES)})
    parent_run_id = parent_run.info.run_id

    for prompt_name, sys_prompt in SYSTEM_PROMPT_VARIANTS.items():
        for top_k in TOP_K_VALUES:
            with mlflow.start_run(run_name=f'eval-{prompt_name}-k{top_k}', nested=True):
                mlflow.set_tags({'notebook': '07_nemotron_rag_eval',
                                 'prompt_variant': prompt_name, 'model': MODEL_ID})
                mlflow.log_params({'prompt_variant': prompt_name, 'top_k': top_k,
                                   'temperature': 0.5, 'n_questions': len(EVAL_QUESTIONS)})

                latencies, tokens, lengths = [], [], []

                for q_idx, question in enumerate(EVAL_QUESTIONS):
                    result = rag(question, sys_prompt, top_k=top_k, temperature=0.5)
                    latencies.append(result['latency_ms'])
                    tokens.append(result['usage'].get('total_tokens', 0))
                    lengths.append(len(result['answer']))
                    mlflow.log_metrics({
                        f'q{q_idx+1}_latency_ms'   : result['latency_ms'],
                        f'q{q_idx+1}_total_tokens' : result['usage'].get('total_tokens', 0),
                        f'q{q_idx+1}_answer_length': len(result['answer']),
                    })

                avg_lat = round(sum(latencies) / len(latencies))
                avg_tok = round(sum(tokens) / len(tokens))
                avg_len = round(sum(lengths) / len(lengths))
                mlflow.log_metrics({'avg_latency_ms': avg_lat, 'avg_total_tokens': avg_tok,
                                    'avg_answer_length': avg_len,
                                    'total_tokens_used': sum(tokens)})

                print(f'  eval-{prompt_name}-k{top_k}: lat={avg_lat}ms  tok={avg_tok}  len={avg_len}')

print(f'\nParent run ID: {parent_run_id}')

In [ ]:
eval_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'",
    order_by=['metrics.avg_answer_length DESC'],
)

cols = ['tags.mlflow.runName', 'params.prompt_variant', 'params.top_k',
        'metrics.avg_latency_ms', 'metrics.avg_total_tokens', 'metrics.avg_answer_length']
available = [c for c in cols if c in eval_runs.columns]
print(f'Evaluation results ({len(eval_runs)} configs, sorted by avg_answer_length desc):\n')
print(eval_runs[available].to_string(index=False))

if 'metrics.avg_answer_length' in eval_runs.columns:
    best = eval_runs.iloc[0]
    print(f'\nBest config: prompt={best.get("params.prompt_variant")}, '
          f'top_k={best.get("params.top_k")}')
    print(f'  avg_latency    : {best.get("metrics.avg_latency_ms")} ms')
    print(f'  avg_tokens     : {best.get("metrics.avg_total_tokens")}')
    print(f'  avg_answer_len : {best.get("metrics.avg_answer_length")} chars')

In [ ]:
if len(eval_runs) > 0 and 'metrics.avg_answer_length' in eval_runs.columns:
    plot_df = eval_runs[['params.prompt_variant', 'params.top_k',
                          'metrics.avg_latency_ms', 'metrics.avg_answer_length']].dropna()
    plot_df['params.top_k'] = plot_df['params.top_k'].astype(str)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = {'concise': '#1f77b4', 'detailed': '#ff7f0e', 'structured': '#2ca02c'}

    for pv in plot_df['params.prompt_variant'].unique():
        sub = plot_df[plot_df['params.prompt_variant'] == pv].sort_values('params.top_k')
        axes[0].plot(sub['params.top_k'], sub['metrics.avg_latency_ms'],
                     marker='o', label=pv, color=colors.get(pv, 'grey'))
        axes[1].plot(sub['params.top_k'], sub['metrics.avg_answer_length'],
                     marker='s', label=pv, color=colors.get(pv, 'grey'))

    for ax, title, ylabel in zip(axes,
                                  ['Latency vs top_k', 'Answer Length vs top_k'],
                                  ['Avg Latency (ms)', 'Avg Answer Length (chars)']):
        ax.set_xlabel('top_k'); ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle(f'RAG Configuration Comparison — Nemotron', fontsize=12)
    plt.tight_layout()
    plt.show()

## Section 5 — Log Best Configuration as Artifact

Store the winning system prompt and RAG parameters as MLflow artifacts.
A serving pipeline can download them with `client.download_artifacts()` — no hardcoded prompts.

In [ ]:
if len(eval_runs) > 0 and 'params.prompt_variant' in eval_runs.columns:
    best_run = eval_runs.iloc[0]
    best_prompt_variant = best_run.get('params.prompt_variant', 'detailed')
    best_top_k          = int(best_run.get('params.top_k', 3))
else:
    best_prompt_variant, best_top_k = 'detailed', 3

best_system_prompt = SYSTEM_PROMPT_VARIANTS.get(best_prompt_variant, DEFAULT_SYSTEM_PROMPT)

rag_config = {
    'model'         : MODEL_ID,
    'endpoint'      : LLM_BASE_URL,
    'system_prompt' : best_system_prompt,
    'top_k'         : best_top_k,
    'temperature'   : 0.5,
    'max_tokens'    : 400,
    'retriever'     : 'tfidf',
    'selected_from' : f'parent_run_id={parent_run_id}',
}

with mlflow.start_run(run_name='best-rag-config') as run:
    mlflow.set_tags({'notebook': '07_nemotron_rag_eval', 'config_type': 'rag_config',
                     'prompt_variant': best_prompt_variant})
    mlflow.log_params({'prompt_variant': best_prompt_variant, 'top_k': best_top_k})

    with tempfile.TemporaryDirectory() as tmp:
        cfg_path    = os.path.join(tmp, 'rag_config.json')
        prompt_path = os.path.join(tmp, 'system_prompt.txt')
        with open(cfg_path, 'w') as f:    json.dump(rag_config, f, indent=2)
        with open(prompt_path, 'w') as f: f.write(best_system_prompt)
        mlflow.log_artifact(cfg_path,    artifact_path='config')
        mlflow.log_artifact(prompt_path, artifact_path='config')

    best_config_run_id = run.info.run_id

print(f'Best RAG config logged.')
print(f'  Run ID         : {best_config_run_id}')
print(f'  Prompt variant : {best_prompt_variant}')
print(f'  top_k          : {best_top_k}')
print()
print('To download in a serving pipeline:')
print(f'  client = mlflow.tracking.MlflowClient()')
print(f'  client.download_artifacts("{best_config_run_id}", "config", "/tmp")')

## Summary

| Feature | API / Pattern |
|---------|---------------|
| TF-IDF retrieval | `TfidfVectorizer` + `cosine_similarity` — no external deps |
| Full trace hierarchy | `@mlflow.trace` on `retrieve`, `generate`, and `rag` |
| Systematic grid eval | `mlflow.start_run(nested=True)` per (prompt × top_k) config |
| Aggregate metrics | `mlflow.log_metrics()` after all questions in a run |
| Compare configs | `mlflow.search_runs(filter_string='parentRunId=...')` |
| Persist best config | `mlflow.log_artifact()` for JSON config + prompt text |

**Dashboard:** `Experiments > rhoai-nemotron-rag-eval`
→ Open `rag-eval-sweep` → select all child runs → **Compare** → Parallel coordinates

**In production, replace:**
- TF-IDF → pgvector / Milvus / OpenSearch vector store
- Static KB → indexed document corpus
- Answer length → LLM-as-judge or domain-specific evaluation metric